# 📡 Starlink Phased Array (AESA) & Capon MVDR Null-Steering
## Interactive Engineering Notebook: 2D Array Factor & Electronic Warfare Jammer Nulling

> **Lead Disciplines:** `radio-frequency-specialist` & `telecom-mathematics-expert`  
> **Standards Reference:** StarSec CSF v1.0 (RFC-003), Technical Monograph Chapter 06 & R&D Whitepaper 02

This notebook simulates the 1,280-element Ku-band Active Electronically Scanned Array (AESA) antenna used in SpaceX Starlink User Terminals (Dishy). It computes the 2D Array Factor $AF(\theta, \phi)$ and evaluates Capon Minimum Variance Distortionless Response (MVDR) adaptive beamforming to place $>45\text{ dB}$ spatial nulls against electronic warfare jammers.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure local simulations module is importable
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from simulations.phased_array_beamforming import (
    PhasedArrayGeometry,
    JammerEmitter,
    compute_steering_weights,
    calculate_array_factor,
    capon_mvdr_beamforming
)

print("✅ Successfully imported Phased Array Electromagnetics Suite")

### 1. Dishy AESA Antenna Geometry

Starlink User Terminals utilize a circular/rectangular Ku-band aperture with $\approx 1,280$ radiating patch antenna elements operating at $12.0\text{ GHz}$ (wavelength $\lambda \approx 2.5\text{ cm}$) spaced at $d = \lambda / 2$ to suppress grating lobes.

In [ ]:
array = PhasedArrayGeometry(
    num_elements_x=32,
    num_elements_y=40,
    element_spacing_x_m=0.0125,   # lambda / 2 at 12 GHz
    element_spacing_y_m=0.0125,
    carrier_frequency_hz=12.0e9
)

print(f"Dishy Ku-Band AESA Array Parameters:")
print(f"  • Total Elements (N)  : {array.total_elements}")
print(f"  • Operating Frequency : {array.carrier_frequency_hz/1e9:.1f} GHz")
print(f"  • Wavelength (λ)      : {array.wavelength_m*1000:.2f} mm")
print(f"  • Aperture Dimensions : {32*1.25:.1f} cm x {40*1.25:.1f} cm")

### 2. 2D Array Factor & Main Beam Steering

The 2D Array Factor is computed by summing the phase-steered element contributions:

$$AF(\theta, \phi) = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} w_{mn} e^{j k (m d_x \sin\theta\cos\phi + n d_y \sin\theta\sin\phi)}$$

In [ ]:
# Steer main beam to satellite look angle (theta = 30°, phi = 0°)
weights_steered = compute_steering_weights(array, theta_steer_deg=30.0, phi_steer_deg=0.0)

theta_scan = np.linspace(-90, 90, 361)
af_pattern_db = [
    calculate_array_factor(array, weights_steered, theta_deg=th, phi_deg=0.0)
    for th in theta_scan
]

plt.figure(figsize=(9, 4.5))
plt.plot(theta_scan, af_pattern_db, 'b-', lw=2, label="Steered Beam (θ₀ = 30°)")
plt.axvline(30.0, color='r', linestyle='--', label="Target Satellite (30°)")
plt.axhline(-3.0, color='gray', linestyle=':', label="HPBW Threshold (-3 dB)")
plt.ylim(-40, 2)
plt.title("Starlink Dishy AESA Normalized Radiation Pattern")
plt.xlabel("Elevation Scan Angle θ (degrees)")
plt.ylabel("Normalized Array Factor (dB)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 3. Capon MVDR Spatial Null-Steering against Hostile EW Jamming

When a ground-based Electronic Warfare jammer transmits high-power RF interference from direction $\theta_{\text{jam}} = -20^\circ$, Capon MVDR adaptively reweights elements using sample covariance matrix $\mathbf{R}_{xx}$:

$$\vec{w}_{\text{opt}} = \frac{\mathbf{R}_{xx}^{-1} \vec{a}(\theta_0)}{\vec{a}^H(\theta_0) \mathbf{R}_{xx}^{-1} \vec{a}(\theta_0)}$$

In [ ]:
# Jammer located at theta = -20°, power = 100 W (+50 dB INR)
jammer = JammerEmitter(theta_deg=-20.0, phi_deg=0.0, power_watts=100.0)

weights_mvdr, null_depth_db = capon_mvdr_beamforming(
    geometry=array,
    desired_theta_deg=30.0,
    desired_phi_deg=0.0,
    jammers=[jammer],
    noise_variance_watts=1e-3
)

print(f"Capon MVDR Beamforming Result:")
print(f"  • Main Beam Target      : +30.0°")
print(f"  • Jammer Azimuth        : -20.0°")
print(f"  • Synthesized Null Depth: {null_depth_db:.2f} dB (Cancellation)")

# Compare standard beamforming vs adaptive nulling pattern
af_mvdr_db = [
    calculate_array_factor(array, weights_mvdr, theta_deg=th, phi_deg=0.0)
    for th in theta_scan
]

plt.figure(figsize=(9, 4.5))
plt.plot(theta_scan, af_pattern_db, 'b--', lw=1.5, label="Quiescent Steering (No Jammer Defense)")
plt.plot(theta_scan, af_mvdr_db, 'r-', lw=2, label=f"Capon MVDR Adaptive Nulling ({null_depth_db:.1f} dB Null)")
plt.axvline(30.0, color='blue', linestyle=':', label="Target Satellite (+30°)")
plt.axvline(-20.0, color='red', linestyle='--', label="Hostile Jammer (-20°)")
plt.ylim(-60, 2)
plt.title("Capon MVDR Spatial Null-Steering Defense against EW Jammer")
plt.xlabel("Elevation Scan Angle θ (degrees)")
plt.ylabel("Normalized Array Factor (dB)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()